In [1]:
import json
import sys
import os
from pathlib import Path
import time
import requests

# Navigate from backend/scripts to backend (project root)
CURRENT_DIR = os.getcwd()
sys.path.append(str(Path(CURRENT_DIR).parent))

from app.db.base import get_db
from app.utils.wichart import getNonce, getHeaders, getToken, getSign, decrypt
from app.services.financial_data_importer import FinancialDataImporter
from app.db.models.financial import ItemValue, Company

db = next(get_db())

sqlite:////Users/phuchdh/Documents/Work/all-in-one-portfolio/backend/portfolio.db


In [5]:

test_symbols = ['LCG', 'FCN']

base_url = "https://wichart.vn/wichartapi/wichart/company/fs"
token = getToken()

importer = FinancialDataImporter(db)

for symbol in test_symbols:
    
    ## query company by ticker
    company = db.query(Company).filter(Company.ticker == symbol).first()
    if not company:
        print(f"Company not found for {symbol}")
        continue
    
    ## Check if data already exists
    data = db.query(ItemValue).filter(ItemValue.company_id == company.company_id).first()
    print(data)
    if data:
        print(f"Data already exists for {symbol}")
        continue
    
    ## Crawl fundamental data
    nonce = getNonce()
    stime = int(time.time() * 1000)

    query_params = {
        "code": symbol,
        "page": 1,
        "type": "quarter",
        "unit": "ty",
        "currency": "vnd",
        "quarter": 1,
    }

    sign_data = {
        'code': symbol,
        'currency': query_params['currency'],
        'nonce': nonce,
        'page': query_params['page'],
        'quarter': query_params['quarter'],
        'sign-token': 'ObBeWhVmYs3tP2Nz$C$FJ@P4AQfTjlPX',
        'stime': stime,
        'type': query_params['type'],
        'unit': query_params['unit'],
        'v': 'v1',
    }

    hashCode = getSign(sign_data)
    response = requests.get(base_url, params=query_params, headers=getHeaders(token, nonce, hashCode, stime))
    enc = response.json()['enc']
    data = json.loads(decrypt(enc))

    try:
        importer.import_financial_data(data, symbol)
        db.commit()
        print(f"Successfully imported financial data for {symbol}")
    except Exception as e:
        print(f"Import failed: {e}")
        raise
    

2025-09-02 23:51:18,134 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-09-02 23:51:18,135 INFO sqlalchemy.engine.Engine SELECT company.company_id AS company_company_id, company.ticker AS company_ticker, company.name AS company_name 
FROM company 
WHERE company.ticker = ?
 LIMIT ? OFFSET ?
2025-09-02 23:51:18,136 INFO sqlalchemy.engine.Engine [cached since 1449s ago] ('LCG', 1, 0)
2025-09-02 23:51:18,141 INFO sqlalchemy.engine.Engine SELECT item_value.item_value_id AS item_value_item_value_id, item_value.item_id AS item_value_item_id, item_value.period_id AS item_value_period_id, item_value.company_id AS item_value_company_id, item_value.value AS item_value_value 
FROM item_value 
WHERE item_value.company_id = ?
 LIMIT ? OFFSET ?
2025-09-02 23:51:18,142 INFO sqlalchemy.engine.Engine [cached since 1449s ago] (98, 1, 0)
None
2025-09-02 23:51:18,364 INFO sqlalchemy.engine.Engine 
            SELECT company_id FROM company WHERE ticker = ?
        
2025-09-02 23:51:18,364 INFO sqlalche